# Aerial Object Classification & Detection
## Notebook 3 — Transfer Learning with EfficientNetB0

Since Custom CNN gave only 67% accuracy, I decided to use Transfer Learning.
I chose EfficientNetB0 because it is lightweight, fast and pre-trained on ImageNet
which contains millions of images — so the model already knows how to detect features.

**Key learning:** I found out that EfficientNetB0 has its own preprocessing function
and using rescale=1./255 with it causes the model to not learn at all!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print('GPU:', tf.config.list_physical_devices('GPU'))

# using 224x224 because EfficientNetB0 was pretrained on this size
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 20

TRAIN_DIR = '/content/dataset/train/'
VALID_DIR = '/content/dataset/valid/'
TEST_DIR  = '/content/dataset/test/'


In [ ]:
# if this shows missing, need to copy from Drive again
for split in ['train', 'valid', 'test']:
    for cls in ['bird', 'drone']:
        path = f'/content/dataset/{split}/{cls}'
        if os.path.exists(path):
            print(f'OK {split}/{cls}: {len(os.listdir(path))} images')
        else:
            print(f'MISSING {split}/{cls} — need to copy from Drive!')

Data Generators with EfficientNet Preprocessing


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

# using EfficientNet's own preprocessor — this is important!
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    shear_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=True
)
val_gen = val_test_datagen.flow_from_directory(
    VALID_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)

print('Class mapping:', train_gen.class_indices)

 Build EfficientNetB0 Model

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

# load pretrained base — exclude the top classification layer
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# freezing base model — dont want to change the pretrained weights yet
base_model.trainable = False

# adding my own classification head
inputs  = tf.keras.Input(shape=(224, 224, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.4)(x)
x       = layers.Dense(128, activation='relu')(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)  # binary output

model_eff = models.Model(inputs, outputs)
model_eff.summary()

Train with Frozen Base


In [ ]:
model_eff.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

os.makedirs('/content/drive/MyDrive/aerial_project/models/', exist_ok=True)
save_path = '/content/drive/MyDrive/aerial_project/models/efficientnet_best.h5'

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(save_path, monitor='val_accuracy',
                                       save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                         patience=3, min_lr=1e-7, verbose=1)
]

print('Phase 1: Training with frozen base...\n')
history1 = model_eff.fit(
    train_gen, epochs=10,
    validation_data=val_gen,
    callbacks=callbacks, verbose=1
)
print(f'\nPhase 1 done! Best val accuracy: {max(history1.history["val_accuracy"])*100:.2f}%')

Fine Tuning


In [ ]:
# unfreeze top 30 layers for fine tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# using a much smaller learning rate to avoid destroying pretrained weights
model_eff.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

print('Phase 2: Fine tuning top layers...\n')
history2 = model_eff.fit(
    train_gen, epochs=10,
    validation_data=val_gen,
    callbacks=callbacks, verbose=1
)
print(f'\nFine tuning done! Best val accuracy: {max(history2.history["val_accuracy"])*100:.2f}%')

Plot Training History

In [ ]:
acc      = history1.history['accuracy']     + history2.history['accuracy']
val_acc  = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss     = history1.history['loss']         + history2.history['loss']
val_loss = history1.history['val_loss']     + history2.history['val_loss']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(acc,     label='Train Accuracy', color='steelblue')
axes[0].plot(val_acc, label='Val Accuracy',   color='tomato')
axes[0].axvline(x=len(history1.history['accuracy']),
                color='green', linestyle='--', label='Fine tune start')
axes[0].set_title('EfficientNetB0 — Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(loss,     label='Train Loss', color='steelblue')
axes[1].plot(val_loss, label='Val Loss',   color='tomato')
axes[1].axvline(x=len(history1.history['loss']),
                color='green', linestyle='--', label='Fine tune start')
axes[1].set_title('EfficientNetB0 — Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

plt.suptitle('EfficientNetB0 Training History', fontsize=14)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/aerial_project/models/efficientnet_plot.png')
plt.show()
print('Plot saved!')

Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

test_gen.reset()
preds       = model_eff.predict(test_gen, verbose=1)
pred_labels = (preds > 0.5).astype(int).flatten()
true_labels = test_gen.classes

print('\nClassification Report:')
print(classification_report(true_labels, pred_labels,
                            target_names=['Bird', 'Drone']))

cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Bird', 'Drone'],
            yticklabels=['Bird', 'Drone'])
plt.title('EfficientNetB0 — Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/aerial_project/models/efficientnet_confusion_matrix.png')
plt.show()

In [ ]:
test_loss, test_acc, test_prec, test_rec = model_eff.evaluate(test_gen, verbose=0)
f1 = 2 * (test_prec * test_rec) / (test_prec + test_rec + 1e-7)

print('=' * 40)
print('EFFICIENTNETB0 — FINAL RESULTS')
print('=' * 40)
print(f'Test Accuracy  : {test_acc*100:.2f}%')
print(f'Test Precision : {test_prec*100:.2f}%')
print(f'Test Recall    : {test_rec*100:.2f}%')
print(f'F1 Score       : {f1*100:.2f}%')
print(f'Test Loss      : {test_loss:.4f}')
print('=' * 40)
print()